#Simple gen ai app using langsmith

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
#Langsmith tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
##Data Ingestion - from the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader


In [ ]:
loader = WebBaseLoader("link")
loader

In [ ]:
docs = loader.load()
docs

In [ ]:
# Load Data-->Docs-->Divide our documents into chunks-->text-->Vectors Embedding--> Vector Store db
#divide the data into chunks
from langchain_text_splitters import RecursiveCharacter_Text_Splitter

text_splitter = RecursiveCharacter_Text_Splitter(chunk_size=1000,chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [ ]:
documents

In [ ]:
#convert these chunks into vector embeddings
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
#store these embeddings in vector db 
from langchain_community.vectorstores import FAISS
vector_storedb = FAISS.from_documents(documents,embeddings)


In [ ]:
vector_storedb

In [ ]:
#Query from a vector db
query="Something that you wanna search"
result=vector_storedb.similarity_search()
result[0].page_content 

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="")


In [ ]:
#Retrieval Chain, Document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following based only on the provided context:
<context>
{context}
</context>
"""
)
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

In [ ]:
from langchain_core.documents import Document
document_chain.invoke ({
    "input":"Something that you wanna search"
    "context":[Document(page_context="Something that you wanna search and something more")]
})

However we want the documents to first come from the retriever we just setup. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question. 

In [ ]:
#Input-->Retriever-->vectorstoredb

retriever=vector_storedb.as_retriever
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [ ]:
##Get the response from the LLM
response=retrieval_chain.invoke({"input":"Something that you wanna search"})
response
response['answer']